## Just OpenAI Client

In [9]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Скрипт для тестирования подключения к локальной модели LLM через OpenAI API
Использует настройки из файла .env
"""

import os
from openai import OpenAI


# Инициализация клиента OpenAI с настройками из .env
client = OpenAI(
    api_key="key",
    base_url="http://81.94.156.140:8999/v1",
)
print(client.models.list())

SyncPage[Model](data=[Model(id='LLM', created=1741978796, object='model', owned_by='vllm', root='LLM', parent=None, max_model_len=9000, permission=[{'id': 'modelperm-f86ab36cdbd247dbb06aca2b103eedf6', 'object': 'model_permission', 'created': 1741978796, 'allow_create_engine': False, 'allow_sampling': True, 'allow_logprobs': True, 'allow_search_indices': False, 'allow_view': True, 'allow_fine_tuning': False, 'organization': '*', 'group': None, 'is_blocking': False}])], object='list')


In [10]:
print("Отправка запроса к модели...")
response = client.chat.completions.create(
    model="LLM",
    messages=[
        {"role": "system", "content": "Ты полезный ассистент."},
        {"role": "user", "content": "Привет! Расскажи что-нибудь интересное о России."}
    ],
    max_tokens=500,
    temperature=0.3,
)

# Вывод ответа модели
print("\n" + "=" * 40)
print("ОТВЕТ МОДЕЛИ:")
print("=" * 40)
print(response.choices[0].message.content)
print("=" * 40 + "\n")

# Вывод информации о запросе
print("Информация о запросе:")
print(f"ID запроса: {response.id}")
print(f"Модель: {response.model}")
print(f"Использовано токенов: {response.usage}")

Отправка запроса к модели...

ОТВЕТ МОДЕЛИ:
Привет! Конечно, расскажу вам интересный факт о России.

Россия — это самая большая страна в мире по площади, занимающая более 17 миллионов квадратных километров. Это означает, что Россия занимает примерно одну десятую часть всей суши нашей планеты. Из-за своих огромных размеров Россия имеет разнообразные климатические зоны, от арктических пустынь на Крайнем Севере до субтропических лесов на южном побережье Крыма.

Одно из самых удивительных явлений в России — это Байкал, самое глубокое озеро в мире. Оно находится в Сибири и известно своей кристально чистой водой и уникальным биоразнообразием. Байкал также является одним из самых древних озер на Земле, его возраст оценивается примерно в 25 миллионов лет.

Россия богата культурным наследием и имеет одну из самых длинных литературных традиций в мире. Такие писатели, как Лев Толстой, Федор Достоевский и Антон Чехов, являются классиками мировой литературы. Россия также славится своими композитора

## With Instructor

In [84]:
from datetime import datetime
from typing import Optional
from pydantic import BaseModel


class TimeFilter(BaseModel):
    start_date: Optional[datetime] = None
    end_date: Optional[datetime] = None

class UserFilter(BaseModel):
    name: Optional[list[str]] = None

class SearchQuery(BaseModel):
    time_filter: TimeFilter
    #user_filter: UserFilter

In [85]:
import instructor

client_instructor = instructor.from_openai(client)
print(client_instructor.models.list())

SyncPage[Model](data=[Model(id='LLM', created=1741983457, object='model', owned_by='vllm', root='LLM', parent=None, max_model_len=9000, permission=[{'id': 'modelperm-d9c9549c5dca48e98f1ba2ced16d9c79', 'object': 'model_permission', 'created': 1741983457, 'allow_create_engine': False, 'allow_sampling': True, 'allow_logprobs': True, 'allow_search_indices': False, 'allow_view': True, 'allow_fine_tuning': False, 'organization': '*', 'group': None, 'is_blocking': False}])], object='list')


In [87]:

# Define hook functions
def log_kwargs(**kwargs):
    print(f"Function called with kwargs: {kwargs}")


def log_exception(exception: Exception):
    print(f"An exception occurred: {str(exception)}")


client_instructor.on("completion:kwargs", log_kwargs)
client_instructor.on("completion:error", log_exception)

print("Отправка запроса к модели...")
print(client_instructor.models.list())
response = client_instructor.chat.completions.create(
    model="LLM",
    response_model=SearchQuery,
    messages=[
        {
            "role": "system",
            "content": f"Now is {datetime.now().strftime('%Y-%m-%d')}",
        },
        {
            "role": "user",
            "content": "Мне надо получить информацию о встрчах с Женей гутиным и Катей за последние 3 месяца",
        },
    ],
    max_tokens=500,
    temperature=0.1,
)

#print("Start date:", response.time_filter.start_date.strftime('%Y-%m-%d') if response.time_filter.start_date else None)
#print("End date:", response.time_filter.end_date.strftime('%Y-%m-%d') if response.time_filter.end_date else None)
#print("User names:", response.user_filter.name)

Отправка запроса к модели...
SyncPage[Model](data=[Model(id='LLM', created=1741983680, object='model', owned_by='vllm', root='LLM', parent=None, max_model_len=9000, permission=[{'id': 'modelperm-a268ada16d5540359b1a856375d671eb', 'object': 'model_permission', 'created': 1741983680, 'allow_create_engine': False, 'allow_sampling': True, 'allow_logprobs': True, 'allow_search_indices': False, 'allow_view': True, 'allow_fine_tuning': False, 'organization': '*', 'group': None, 'is_blocking': False}])], object='list')
Function called with kwargs: {'messages': [{'role': 'system', 'content': 'Now is 2025-03-14'}, {'role': 'user', 'content': 'Мне надо получить информацию о встрчах с Женей гутиным и Катей за последние 3 месяца'}], 'model': 'LLM', 'max_tokens': 500, 'temperature': 0.1, 'tools': [{'type': 'function', 'function': {'name': 'SearchQuery', 'description': 'Correctly extracted `SearchQuery` with all the required parameters with correct types', 'parameters': {'$defs': {'TimeFilter': {'pro

In [101]:

        
tools = [
    {
        "type": "function",
        "function": {
            "name": "retrieve_payment_status",
            "description": "Get payment status of a transaction",
            "parameters": {
                "type": "object",
                "properties": {
                    "transaction_id": {
                        "type": "string",
                        "description": "The transaction id.",
                    }
                },
                "required": ["transaction_id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "retrieve_payment_date",
            "description": "Get payment date of a transaction",
            "parameters": {
                "type": "object",
                "properties": {
                    "transaction_id": {
                        "type": "string",
                        "description": "The transaction id.",
                    }
                },
                "required": ["transaction_id"],
            },
        },
    }
]

In [122]:
response = client.chat.completions.create(
            model="LLM", 
            messages=[{"role": "user", "content": "What's the status of my transaction with ID = T1001?"}],
            tools=tools,
            tool_choice="auto",
            temperature=0.35,
        )

In [123]:
print(response.choices[0].message.tool_calls)
print(response.choices[0].message.content)

[ChatCompletionMessageToolCall(id='37XgYtYs4', function=Function(arguments='{"transaction_id": "T1001"}', name='retrieve_payment_status'), type='function')]
None
